# NB03 - Vues

## Mise en place

Exécuter la cellules suivantes pour s'assurer créer / rafraichir les différents éléments du **Catalogue**

In [0]:
%run "./Resources/NB00/Setup"

Vérifier que l'on utilise le bon **Catalogue** et **Schéma** pour éviter de devoir renseigner le chemin complet dans les requêtes.

`SELECT current_catalog(), current_database();`

et pour changer les catalogues et schémas par défaut utiliser la requête : 

```sql
USE CATALOG labuser_<username>;
USE SCHEMA sql_analytics;
```

In [0]:
%sql
SELECT current_catalog(), current_database();

In [0]:
USE CATALOG labuser_edadou;
USE SCHEMA sql_analytics;

## Introduction sur les Vues

1. Vue Standard
2. Vue Temporaire
3. Vue Matérialisée
4. Vue Dynamique



### 1. Vue Standard

Une **vue standard** est une définition logique sauvegardée dans le catalogue. Elle ne stocke aucune donnée : à chaque consultation, la requête sous-jacente est ré-exécutée.

**Création :**

```sql
CREATE VIEW ma_vue AS
SELECT colonne1, colonne2
FROM ma_table
WHERE condition;
```

**Points clés :**

- Toujours à jour (ré-exécute la requête à chaque accès)
- Aucune donnée stockée → pas de coût de stockage
- Idéale pour réutiliser une même logique de filtrage ou de jointure
- Persistée dans Unity Catalog, peut être partagée et sécurisée

In [0]:
create or replace view sales_rep_vw as 
select
    *,
    'Standard View' as view_type
from
    ca_sales_reps;

In [0]:
show tables;

Avec `show tables;` la vue est visible dans le schéma : 

![image_1790934646300.png](./Resources/image_1790934646300.png "image_1790934646300.png")

On peut également avoir également les informations (uniquement pour les vues) avec `show views` et avoir une déscription détaillée de la vue avec le `describe extended <view_name>`

In [0]:
describe extended sales_rep_vw;

In [0]:
select * from sales_rep_vw;

In [0]:
insert into ca_sales_reps values ('SR013', 'Louis Vuitton', 'Ouest', '2017-01-01', 'louis.vuitton@databricks.com', true);

Si on insert une nouvelle ligne dans la table source, et que on requête la vue, les données seront à jours :

In [0]:
select * from sales_rep_vw;

### 2. Vue Temporaire

Une **vue temporaire** (ou *temp view*) existe uniquement pour la durée de la session ou du cluster. Elle n'est **pas persistée** dans le catalogue Unity et n'est visible que dans la session courante.

**Création :**

```sql
CREATE TEMP VIEW ma_vue_temp AS
SELECT colonne1, colonne2
FROM ma_table
WHERE condition;
```

**Points clés :**

- Limitée à la session en cours — disparaît à la déconnexion
- Non enregistrée dans le catalogue → invisible avec `show tables`
- Idéale pour des transformations intermédiaires ou des tests rapides
- Pratique pour partager une logique entre plusieurs requêtes d'un même notebook sans polluer le schéma



In [0]:
create or replace temporary view sales_rep_tmp_vw as 
select *,
'Temporary View' as view_type
from ca_sales_reps;

In [0]:
show views;

les vues temporaire n'ont pas de **namespace** car elles ne sont pas stockées : 

![image_1790935996242.png](./Resources/image_1790935996242.png "image_1790935996242.png")

Si le **warehouse** ou **cluster** est détaché / éteint, ... la vue sera supprimée

In [0]:
select * from sales_rep_tmp_vw;

### 3. Vue Matérialisée

Une **vue matérialisée** est une vue dont le **résultat est calculé et stocké physiquement** dans Unity Catalog. Contrairement à une vue standard, les données sont persistées et mises à jour (rafraîchies) selon une stratégie définie — elles ne sont **pas** recalculées à chaque consultation.

**Création :**

```sql
CREATE MATERIALIZED VIEW ma_vue_mat AS
SELECT colonne1, colonne2, SUM(colonne3) AS total
FROM ma_table
WHERE condition
GROUP BY colonne1, colonne2;
```

**Rafraîchissement (refresh) :**

Les données de la vue matérialisée ne sont pas mises à jour automatiquement à chaque accès. Il faut déclencher un **refresh** :

```sql
REFRESH MATERIALIZED VIEW ma_vue_mat;
```

On peut aussi planifier les rafraîchissements directement dans la définition :

```sql
-- Rafraîchissement automatique quand les données en amont changent (recommandé)
CREATE MATERIALIZED VIEW ma_vue_mat
TRIGGER ON UPDATE
AS SELECT ... ;

-- Rafraîchissement périodique fixe
CREATE MATERIALIZED VIEW ma_vue_mat
SCHEDULE EVERY 1 HOUR
AS SELECT ... ;

-- Rafraîchissement avec expression CRON
CREATE MATERIALIZED VIEW ma_vue_mat
SCHEDULE CRON '0 0 8 * * ? *' AT TIME ZONE 'UTC'
AS SELECT ... ;
```

**Stratégies de rafraîchissement :**

| Stratégie | Description |
|---|---|
| `TRIGGER ON UPDATE` | Rafraîchit automatiquement quand les données sources changent. Latence la plus faible, pas de refresh inutile. |
| `SCHEDULE EVERY` | Rafraîchissement périodique fixe (1 HOUR minimum). |
| `SCHEDULE CRON` | Expression cron pour un scheduling précis. Toujours préciser `AT TIME ZONE`. |

**Points clés :**

- Les données sont **stockées physiquement** → coût de stockage, mais performances de lecture bien meilleures
- Idéale pour des agrégations, jointures complexes ou des dashboards qui doivent être rapides
- Le rafraîchissement peut être **manuel** (`REFRESH MATERIALIZED VIEW`), **automatique** (`TRIGGER ON UPDATE`) ou **planifié** (`SCHEDULE`)
- Persistée dans Unity Catalog, peut être partagée et sécurisée
- Les commandes `OPTIMIZE` et `VACUUM` ne sont pas disponibles — la maintenance est gérée automatiquement par Predictive Optimization
- `CLUSTER BY` est recommandé pour optimiser le filtrage (plutôt que `PARTITIONED BY`)

**Comparaison rapide avec une vue standard :**

| | Vue standard | Vue matérialisée |
|---|---|---|
| Données stockées | Non | Oui |
| À jour à chaque accès | Oui | Non (dépend du refresh) |
| Coût de stockage | Aucun | Oui |
| Performance de lecture | Dépend de la requête | Rapide (données pré-calculées) |


> **Attention** Les vues matérialisées ne sont pas autorisées sur les clusters / warehouses **Serverless** 



In [0]:
create or replace materialized view sales_rep_mv as 
select *,
'Materialized View' as view_type
from ca_sales_reps;

Si on fait un `SHOW VIEWS`

la colonne **isMaterialized** sera a `true`

In [0]:
select * from sales_rep_mv;

Si on fait un insert dans la table source `ca_sales_reps`, et que on refait un select sur la vue la nouvelle ligne ne sera pas visible

In [0]:
insert into ca_sales_reps values ('SR014', 'Christian Dior', 'Est', '2018-01-01', 'christian.dior@databricks.com', true);

In [0]:
select * from sales_rep_mv;

Pour avoir accès aux nouvelles données il faut rafraichir la vue : 

In [0]:
refresh materialized view sales_rep_mv;

select * from sales_rep_mv;

On peut automatiser le **refresh** de la vue avec : 

In [0]:
create or replace materialized view sales_rep_mv_scheduled
schedule every 1 hour
as 
select *,
'Materialized View' as view_type
from ca_sales_reps;

In [0]:
describe extended sales_rep_mv_scheduled;

Avec la description de la vue on a également les informations quant au **refresh** de la vue : 
- *Refresh Information*
- les informations des derniers *Refresh*
- le *Schedule* de la vue

### 4. Vue Dynamique

Une **vue dynamique** (ou *dynamic view*) est une vue standard qui intègre des fonctions de sécurité comme `current_user()` ou `is_member()` pour appliquer un **filtrage au niveau des lignes** (row-level security) et/ou un **masquage au niveau des colonnes** (column-level security), en fonction de l'utilisateur qui l'interroge.

Contrairement à une vue classique, le résultat renvoyé **change dynamiquement selon l'identité de l'utilisateur connecté**. Un même `SELECT *` ne retournera pas les mêmes lignes ou colonnes selon qui l'exécute.

**Création :**

```sql
CREATE OR REPLACE VIEW sales_rep_secure_vw AS
SELECT
  sales_rep_id,
  sales_rep_name,
  region,
  hire_date,
  email,
  is_active,
  -- Masquage de l'email selon l'appartenance au groupe
  CASE
    WHEN is_member('hr_managers') THEN email
    ELSE '****@databricks.com'
  END AS masked_email,
  -- Filtrage des lignes selon l'utilisateur
  CASE
    WHEN is_member('sales_managers') THEN true
    WHEN region = current_user() THEN true
    ELSE false
  END AS can_see
FROM ca_sales_reps
WHERE
  is_member('sales_managers')
  OR sales_rep_name = current_user();
```

**Fonctions clés :**

| Fonction | Rôle |
|---|---|
| `current_user()` | Retourne l'email/identité de l'utilisateur courant |
| `is_member('groupe')` | Retourne `true` si l'utilisateur appartient au groupe Unity Catalog spécifié |

**Points clés :**

- La sécurité est appliquée **au moment de la requête** — aucun rafraîchissement nécessaire
- Permet de partager une **même table** entre plusieurs utilisateurs avec des niveaux d'accès différents
- Les utilisateurs ne peuvent **pas contourner** le filtrage : la vue est la seule porte d'entrée
- Idéale pour implémenter des politiques de gouvernance sans dupliquer les données
- Fonctionne avec Unity Catalog (groupes, comptes, utilisateurs)

**Comparaison avec les autres types de vues :**

| | Vue standard | Vue temporaire | Vue matérialisée | Vue dynamique |
|---|---|---|---|---|
| Persistée dans le catalogue | Oui | Non | Oui | Oui |
| Données stockées physiquement | Non | Non | Oui | Non |
| Sécurité row/column-level | Non | Non | Non | **Oui** |
| À jour à chaque accès | Oui | Oui | Non (refresh) | Oui |

> **Attention** La vue dynamique est une vue standard : la sécurité repose sur la **logique SQL** de la vue. Assurez-vous que les utilisateurs n'ont **pas** de privilèges `SELECT` directs sur la table source — sinon ils pourraient contourner la vue.



In [0]:
select is_account_group_member('admins');

Le compte n'appartient pas au groupe **admins** et on va filtrer les valeurs au niveau de la : 
- colonne
- ligne

Dans un premier temps on va créer une vue qui dans le cas où on ne fait pas partie du groupe **admins**, les adresses mails sont masquées :

In [0]:
CREATE OR REPLACE VIEW sales_rep_secure_vw AS
SELECT
  salesrep_id,
  salesrep_name,
  region,
  hire_date,
  active,
  -- Masquage de l'email selon l'appartenance au groupe
  CASE
    WHEN is_account_group_member('admins') THEN email
    ELSE '****@****.**'
  END AS masked_email
FROM ca_sales_reps;

In [0]:
select * from sales_rep_secure_vw;

Maintenant on va créer une vue avec les lignes filtrées au niveau de la région en fonction de l'appartenance a un groupe : 

In [0]:
select is_account_group_member('admins_ouest');

In [0]:
CREATE OR REPLACE VIEW sales_rep_secure_rows_vw AS
SELECT
  salesrep_id,
  salesrep_name,
  region,
  hire_date,
  active,
  email
FROM ca_sales_reps
where 
    case 
        when  is_account_group_member('admins_ouest') then true
        else region = 'Ouest'
    end;

In [0]:
select * from sales_rep_secure_rows_vw;